In [1]:
%load_ext autoreload
%autoreload 2

In [7]:
import pandas as pd
import numpy as np
from preprocessing.ecg_preprocessor import ECGPreprocessor
from optimization.fmm_optimizer import FMMOptimizer


In [8]:
test_ecg_path = "~/UNI/CORSI/Visione_Artificiale/progetto/datasets/PTB/s0001_re.csv"
lead_names = ["i","ii","iii","avl","avr","avf","v1","v2","v3","v4","v5","v6"]
df = pd.read_csv(test_ecg_path)[lead_names]
# shape = (n_leads, n_samples)
data_in = df.to_numpy().T

In [9]:
# PREPROCESSING

sr = 500
preprocessor = ECGPreprocessor(sr)
results_post_filters, m_detrend, error_preprocessing = preprocessor.run(data_in)
results_post_filters

array([{'loessRPeaksEnd': array([  302.,  1127.,  1922.,  2727.,  3540.,  4397.,  5246.,  6064.,
               6921.,  7777.,  8602.,  9440., 10272., 11111., 11925., 12756.,
              13581., 14404., 15215., 16041., 16865., 17691., 18504., 19335.,
              20169., 21007., 21836., 22679., 23501., 24320., 25142., 25977.,
              26795., 27605., 28422., 29260., 30093., 30903., 31714., 32558.,
              33396., 34206., 35037., 35865., 36689., 37503.]), 'loessSegEnd': array([[    0,   797],
              [  798,  1604],
              [ 1605,  2405],
              [ 2406,  3215],
              [ 3216,  4055],
              [ 4056,  4907],
              [ 4908,  5737],
              [ 5738,  6579],
              [ 6580,  7435],
              [ 7436,  8272],
              [ 8273,  9105],
              [ 9106,  9940],
              [ 9941, 10776],
              [10777, 11600],
              [11601, 12424],
              [12425, 13251],
              [13252, 14075],
         

In [10]:
# estrazione del primo battito

beat_idx = 0
beat_start_idx = int(results_post_filters[0]['loessSegEnd'][beat_idx, 0])
beat_end_idx = int(results_post_filters[0]['loessSegEnd'][beat_idx, 1])

v_data_matrix_beat = m_detrend[:, beat_start_idx:beat_end_idx].T
annotation_R = int(results_post_filters[0]['loessRPeaksEnd'][beat_idx])
n_obs = v_data_matrix_beat.shape[0]

v_data_matrix_beat[0]

array([-0.07996017, -0.04532229,  0.03463657, -0.05755077,  0.06289142,
       -0.00535596,  0.02838667, -0.01017926, -0.01859679, -0.04875561,
       -0.04742871, -0.02670766])

In [11]:
# OPTIMIZATION

n_waves = 5
optimizer = FMMOptimizer(n_waves=n_waves, max_iter=10)

print(f"Backfitting sul battito {beat_idx}")

params_per_lead, fitted_waves = optimizer.fit(
    ecg_data=v_data_matrix_beat, 
    annotation=annotation_R,
)

lead = 0
print(f"Parametri trovati per derivazione {lead}")

for key, value in params_per_lead[lead].items():
    print(f"Parametro {key}: ")
    print(f"  - Onda P: {value[0]}")
    print(f"  - Onda Q: {value[1]}")
    print(f"  - Onda R: {value[2]}")
    print(f"  - Onda S: {value[3]}")
    print(f"  - Onda T: {value[4]}")


Backfitting sul battito 0
Iterazione 1 ciclo principale
Onda 1 ciclo principale
fitted_waves [[[0. 0. 0. 0. 0.]
  [0. 0. 0. 0. 0.]
  [0. 0. 0. 0. 0.]
  ...
  [0. 0. 0. 0. 0.]
  [0. 0. 0. 0. 0.]
  [0. 0. 0. 0. 0.]]

 [[0. 0. 0. 0. 0.]
  [0. 0. 0. 0. 0.]
  [0. 0. 0. 0. 0.]
  ...
  [0. 0. 0. 0. 0.]
  [0. 0. 0. 0. 0.]
  [0. 0. 0. 0. 0.]]

 [[0. 0. 0. 0. 0.]
  [0. 0. 0. 0. 0.]
  [0. 0. 0. 0. 0.]
  ...
  [0. 0. 0. 0. 0.]
  [0. 0. 0. 0. 0.]
  [0. 0. 0. 0. 0.]]

 ...

 [[0. 0. 0. 0. 0.]
  [0. 0. 0. 0. 0.]
  [0. 0. 0. 0. 0.]
  ...
  [0. 0. 0. 0. 0.]
  [0. 0. 0. 0. 0.]
  [0. 0. 0. 0. 0.]]

 [[0. 0. 0. 0. 0.]
  [0. 0. 0. 0. 0.]
  [0. 0. 0. 0. 0.]
  ...
  [0. 0. 0. 0. 0.]
  [0. 0. 0. 0. 0.]
  [0. 0. 0. 0. 0.]]

 [[0. 0. 0. 0. 0.]
  [0. 0. 0. 0. 0.]
  [0. 0. 0. 0. 0.]
  ...
  [0. 0. 0. 0. 0.]
  [0. 0. 0. 0. 0.]
  [0. 0. 0. 0. 0.]]]
----------------------------------------------------------------------
Onda 2 ciclo principale
fitted_waves [[[-0.51200087  0.          0.          0.          0.       